<a href="https://colab.research.google.com/github/gitmystuff/DSChunks/blob/main/Covariance%2C_Regression_Coefficients_and_Multicollinearity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Covariance, Regression Coefficients & Multicollinearity

* With **one predictor**, the regression coefficient has a simple direct formula: $$\beta = \frac{\text{Cov}(X, Y)}{\text{Var}(X)}$$ -- literally a covariance divided by a variance.
* With **multiple predictors**, this generalizes to matrix form: $$\boldsymbol{\beta} = \boldsymbol{\Sigma}_{XX}^{-1}\, \boldsymbol{\Sigma}_{XY}$$ where $\boldsymbol{\Sigma}_{XX}$ is the covariance matrix *among the predictors* and $\boldsymbol{\Sigma}_{XY}$ holds each predictor's covariance with the outcome.
* The **off-diagonal entries of $\boldsymbol{\Sigma}_{XX}$** -- covariances *between* predictors -- get folded in through that matrix inversion. This is what makes multiple regression coefficients "partial": each one reflects a predictor's relationship with $Y$ *after accounting for* its overlap with the other predictors.
* When two predictors are highly correlated (a large off-diagonal entry), $\boldsymbol{\Sigma}_{XX}$ becomes close to singular (determinant near zero), and its inverse becomes unstable. This is **multicollinearity**: coefficients become unreliable, can swing with small changes in the data, and can even take counterintuitive signs.
* **Contrast worth remembering:** in PCA, a large off-diagonal covariance means "these features share a direction, worth blending together." In regression, that same large off-diagonal means "these predictors overlap, their individual coefficients are hard to pin down." Same matrix, same number, different consequence depending on what you're using it for.


## The data

Reusing `years_experience` and `skill_count` from the eigenvectors chunk -- recall these two are strongly correlated (covariance 0.953 once standardized). Here we add a `salary` outcome, built to genuinely depend on both.


In [1]:
import numpy as np

years_experience = np.array([2, 4, 5, 7, 8, 10, 12, 14])
skill_count       = np.array([3, 5, 4, 8, 7, 11, 10, 15])
salary            = np.array([53.7, 65.6, 66.2, 77.9, 83.8, 97.9, 100.9, 120.2])


## Single predictor: the direct formula

Using only `years_experience` to predict `salary`:

$$\beta = \frac{\text{Cov}(\text{years_experience}, \text{salary})}{\text{Var}(\text{years_experience})} = \frac{89.436}{16.786} = 5.328$$


In [2]:
cov_xy = np.cov(years_experience, salary, ddof=1)[0, 1]
var_x = np.var(years_experience, ddof=1)
beta_single = cov_xy / var_x

print(f"cov(years_experience, salary): {cov_xy:.3f}")
print(f"var(years_experience):         {var_x:.3f}")
print(f"beta (hand formula):           {beta_single:.3f}")

# Verify against a standard least-squares fit
A = np.column_stack([np.ones(8), years_experience])
coef, *_ = np.linalg.lstsq(A, salary, rcond=None)
print(f"\nVerify -- intercept: {coef[0]:.3f}, slope: {coef[1]:.3f}")


cov(years_experience, salary): 89.436
var(years_experience):         16.786
beta (hand formula):           5.328

Verify -- intercept: 41.982, slope: 5.328


## Multiple predictors: the matrix formula

Now predict `salary` from **both** `years_experience` and `skill_count` together. First, build $\boldsymbol{\Sigma}_{XX}$ (covariance between the two predictors) and $\boldsymbol{\Sigma}_{XY}$ (each predictor's covariance with salary):


In [3]:
X = np.column_stack([years_experience, skill_count])
Sigma_XX = np.cov(X.T, ddof=1)
Sigma_XY = np.array([
    np.cov(years_experience, salary, ddof=1)[0, 1],
    np.cov(skill_count, salary, ddof=1)[0, 1],
])

print("Sigma_XX (predictor covariance matrix):")
print(np.round(Sigma_XX, 3))
print("\nSigma_XY (each predictor's covariance with salary):")
print(np.round(Sigma_XY, 3))


Sigma_XX (predictor covariance matrix):
[[16.786 15.679]
 [15.679 16.125]]

Sigma_XY (each predictor's covariance with salary):
[89.436 86.611]


Now invert $\boldsymbol{\Sigma}_{XX}$ and multiply by $\boldsymbol{\Sigma}_{XY}$:

$$\boldsymbol{\beta} = \boldsymbol{\Sigma}_{XX}^{-1}\, \boldsymbol{\Sigma}_{XY}$$


In [4]:
beta_multi = np.linalg.inv(Sigma_XX) @ Sigma_XY
print(f"beta (years_experience): {beta_multi[0]:.3f}")
print(f"beta (skill_count):      {beta_multi[1]:.3f}")

# Verify against a standard least-squares fit with both predictors
A2 = np.column_stack([np.ones(8), years_experience, skill_count])
coef2, *_ = np.linalg.lstsq(A2, salary, rcond=None)
print(f"\nVerify -- intercept: {coef2[0]:.3f}, "
      f"years_experience: {coef2[1]:.3f}, skill_count: {coef2[2]:.3f}")


beta (years_experience): 3.389
beta (skill_count):      2.076

Verify -- intercept: 40.662, years_experience: 3.389, skill_count: 2.076


**Notice something important:** `years_experience`'s coefficient dropped from **5.328** (alone) to **3.389** (with `skill_count` included). Neither number is "wrong" -- they're answering different questions. The single-predictor coefficient says "salary rises about $5,328 per year of experience, ignoring everything else." The multi-predictor coefficient says "salary rises about $3,389 per year of experience, *after accounting for* skill count" -- some of what looked like an experience effect was actually overlapping with skill count, and the off-diagonal covariance (15.679) is exactly what caused that adjustment during the matrix inversion.


## Multicollinearity: when the off-diagonal gets too large

`years_experience` and `skill_count` are strongly correlated here, which makes $\boldsymbol{\Sigma}_{XX}$ closer to singular than it would be with two unrelated predictors. Check the determinant and condition number (a measure of how close a matrix is to being singular -- larger means closer to trouble):


In [5]:
det = np.linalg.det(Sigma_XX)
cond = np.linalg.cond(Sigma_XX)
print(f"det(Sigma_XX): {det:.3f}")
print(f"condition number: {cond:.3f}")


det(Sigma_XX): 24.852
condition number: 41.558


On its own that number is hard to interpret -- so let's see the actual practical consequence: refit the regression 8 times, each time leaving out one row, and watch how much the coefficients move around.


In [6]:
print("Leave-one-out coefficients (drop each row once):")
for i in range(8):
    mask = np.ones(8, dtype=bool)
    mask[i] = False
    A_loo = np.column_stack([np.ones(mask.sum()), years_experience[mask], skill_count[mask]])
    coef_loo, *_ = np.linalg.lstsq(A_loo, salary[mask], rcond=None)
    print(f"  drop row {i}: years_experience={coef_loo[1]:.2f}, skill_count={coef_loo[2]:.2f}")


Leave-one-out coefficients (drop each row once):
  drop row 0: years_experience=3.39, skill_count=2.07
  drop row 1: years_experience=3.51, skill_count=1.99
  drop row 2: years_experience=3.35, skill_count=2.13
  drop row 3: years_experience=3.08, skill_count=2.38
  drop row 4: years_experience=3.19, skill_count=2.29
  drop row 5: years_experience=3.44, skill_count=2.01
  drop row 6: years_experience=4.00, skill_count=1.54
  drop row 7: years_experience=3.55, skill_count=1.77


The `years_experience` coefficient swings from about 3.08 up to 4.00, and `skill_count` swings from about 1.54 up to 2.38, just from dropping a *single row* out of eight. That's real instability, directly traceable to the strong overlap (large off-diagonal covariance) between these two predictors -- with less-correlated predictors, dropping one row would barely move the coefficients at all.

**The practical takeaway:** a large off-diagonal covariance isn't inherently bad -- in PCA, it's exactly what lets us blend correlated features into a single, meaningful component. In regression, that same large off-diagonal is a warning sign: it means the model can't cleanly separate the two predictors' individual effects, and the resulting coefficients should be trusted less than their formula-derived precision suggests.
